In [71]:
!pip -q install sacremoses pandas numpy

In [73]:
import os
import re
import math
import random
import tarfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
from sacremoses import MosesTokenizer

random.seed(42)

# 1. Gathering data

I download the corpora data from: https://wortschatz-leipzig.de/de/download, which provides sentence-level corpora for a variety of languages and domains. According to the project website:

 "The German Vocabulary Project provides corpora in various languages and sizes, using the same formats and comparable sources. All data is available as plain text. [...] The corpora contain randomly selected sentences from the respective corpus language. [...] The texts used are always broken down into individual sentences and randomly sorted, making reconstruction of the original text impossible. Ungrammatical sentences and foreign language material have been removed as best as possible. Because information on word co-occurrences is useful for many applications, this information has been pre-calculated and is also included. For each word, the most significant neighborhood (left & right) and sentence co-occurrences are listed. The German Vocabulary Project uses publicly available sources for automatic corpus creation without considering the content of individual sources. No responsibility is assumed for the content of the texts. In particular, the views and opinions contained in the data represent the sole views of the respective authors."

The data I will work with, provided on the abovementioned web page, were scraped from 2016 Wikipedia.

In [9]:
# Keep the same three languages everywhere in the notebook.
LANGUAGES = {
    "English": {
        "url": "https://downloads.wortschatz-leipzig.de/corpora/eng_wikipedia_2016_100K.tar.gz",
        "moses": "en",
    },
    "Polish": {
        "url": "https://downloads.wortschatz-leipzig.de/corpora/pol_wikipedia_2016_100K.tar.gz",
        "moses": "pl",
    },
    "Hungarian": {
        "url": "https://downloads.wortschatz-leipzig.de/corpora/hun_wikipedia_2016_100K.tar.gz",
        "moses": "hu",
    },
}

DOWNLOAD_DIR = "downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Loading the data and preparing the data for analysis
Since the data are provided in a zip, where I only need the "sentence data" and each sentence in the "sentence data" is numbered, I need to further process the data, before I proceed with the analysis.

In [11]:
def download_file(url, output_path):
    if not os.path.exists(output_path):
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, output_path)
    return output_path


def find_sentences_member(tar):
    candidates = [m for m in tar.getmembers() if m.name.endswith("-sentences.txt")]
    if not candidates:
        raise FileNotFoundError("Could not find a *-sentences.txt file in the archive.")
    return candidates[0]


def load_and_clean_sentences(tar_path):
    with tarfile.open(tar_path, "r:gz") as tar:
        member = find_sentences_member(tar)
        with tar.extractfile(member) as f:
            raw_lines = f.read().decode("utf-8", errors="ignore").splitlines()

    sentences = []
    for line in raw_lines:
        line = re.sub(r"^\d+\t", "", line)   # remove sentence IDs from corpora
        line = line.strip().lower()
        line = re.sub(r"\s+", " ", line)
        if line:
            sentences.append(line)

    return sentences


def tokenize_sentences(sentences, moses_lang):
    tokenizer = MosesTokenizer(lang=moses_lang)
    tokenized = []

    for sent in sentences:
        tok = tokenizer.tokenize(sent, return_str=True, escape=False)
        tok = re.sub(r"\s+", " ", tok).strip()
        if tok:
            tokenized.append(tok)

    return tokenized


def split_sentences(sentences, seed=42):
    """
    Split at sentence level so we do not destroy character sequence structure
    inside sentences. We shuffle sentences, not individual tokens.
    """
    sents = list(sentences)
    rnd = random.Random(seed)
    rnd.shuffle(sents)

    n = len(sents)
    train_end = int(0.8 * n)
    heldout_end = int(0.9 * n)

    return {
        "train": sents[:train_end],
        "heldout": sents[train_end:heldout_end],
        "test": sents[heldout_end:],
    }

# 2. Download data, tokenize everything and report the size of the data in tokens and bytes.
We will use Sacremoses library for this task.

In [14]:
tokenized_corpora = {}
size_rows = []

for language, info in LANGUAGES.items():
    tar_path = os.path.join(DOWNLOAD_DIR, os.path.basename(info["url"]))
    download_file(info["url"], tar_path)

    raw_sentences = load_and_clean_sentences(tar_path)
    tokenized_sentences = tokenize_sentences(raw_sentences, info["moses"])
    tokenized_corpora[language] = tokenized_sentences

    full_text = "\n".join(tokenized_sentences)
    token_count = sum(len(sent.split()) for sent in tokenized_sentences)
    byte_count = len(full_text.encode("utf-8"))

    size_rows.append({
        "language": language,
        "sentences": len(tokenized_sentences),
        "tokens": token_count,
        "bytes": byte_count,
    })

sizes_df = pd.DataFrame(size_rows).sort_values("language")
sizes_df

,language,sentences,tokens,bytes
0,English,100000,2387805,13149854
2,Hungarian,100000,1833664,12830423
1,Polish,100000,1751344,11563037


In [16]:
# Print version (in case we do not work with Jupyter notebook)
print("Corpus sizes:\n")
print(sizes_df.to_string(index=False))

# check
for _, row in sizes_df.iterrows():
    assert row["tokens"] >= 200_000, f"{row['language']} has less than 200k tokens." 

Corpus sizes:

 language  sentences  tokens    bytes
  English     100000 2387805 13149854
Hungarian     100000 1833664 12830423
   Polish     100000 1751344 11563037


In [18]:
splits = {
    language: split_sentences(sentences, seed=42)
    for language, sentences in tokenized_corpora.items()
}

for language in LANGUAGES:
    print(language)
    print("  train sentences:  ", len(splits[language]["train"]))
    print("  heldout sentences:", len(splits[language]["heldout"]))
    print("  test sentences:   ", len(splits[language]["test"]))

English
  train sentences:   80000
  heldout sentences: 10000
  test sentences:    10000
Polish
  train sentences:   80000
  heldout sentences: 10000
  test sentences:    10000
Hungarian
  train sentences:   80000
  heldout sentences: 10000
  test sentences:    10000


# 3. Analysis

In [20]:
def build_char_ngram_counts(sentences, n, vocab=None, pad=True):
    """
    Count character n-grams.
    If vocab is given, unseen chars are mapped to <UNK>.
    For bigrams/trigrams we add start/end symbols for language modeling.
    """
    counts = Counter()

    for sent in sentences:
        chars = [ch if (vocab is None or ch in vocab) else "<UNK>" for ch in sent]

        if pad:
            seq = ["<s>"] * (n - 1) + chars + ["</s>"]
        else:
            seq = chars

        for i in range(len(seq) - n + 1):
            counts[tuple(seq[i:i+n])] += 1

    return counts


def build_language_model(train_sentences):
    vocab = sorted(set("".join(train_sentences)))

    # Raw character counts for the LM
    unigram_counts = build_char_ngram_counts(train_sentences, 1, vocab=vocab, pad=False)
    bigram_counts = build_char_ngram_counts(train_sentences, 2, vocab=vocab, pad=True)
    trigram_counts = build_char_ngram_counts(train_sentences, 3, vocab=vocab, pad=True)

    # MLE probabilities
    unigram_total = sum(unigram_counts.values())
    unigram_probs = {
        gram: count / unigram_total
        for gram, count in unigram_counts.items()
    }

    bigram_context_counts = Counter()
    for gram, count in bigram_counts.items():
        bigram_context_counts[(gram[0],)] += count

    bigram_probs = {
        gram: count / bigram_context_counts[(gram[0],)]
        for gram, count in bigram_counts.items()
    }

    trigram_context_counts = Counter()
    for gram, count in trigram_counts.items():
        trigram_context_counts[(gram[0], gram[1])] += count

    trigram_probs = {
        gram: count / trigram_context_counts[(gram[0], gram[1])]
        for gram, count in trigram_counts.items()
    }

    # For add-delta smoothing in the trigram model:
    # possible next symbols = all training chars + </s> + <UNK>
    output_vocabulary_size = len(vocab) + 2

    return {
        "vocab": set(vocab),
        "output_vocabulary_size": output_vocabulary_size,
        "unigram_counts": unigram_counts,
        "bigram_counts": bigram_counts,
        "trigram_counts": trigram_counts,
        "unigram_probs": unigram_probs,
        "bigram_probs": bigram_probs,
        "trigram_probs": trigram_probs,
        "trigram_context_counts": trigram_context_counts,
    }


def raw_trigram_counts(sentences):
    """
    Count raw character trigrams without padding.
    This is what we use for '5 most common character trigrams'.
    """
    counts = Counter()
    total = 0

    for sent in sentences:
        chars = list(sent)
        for i in range(len(chars) - 2):
            trigram = "".join(chars[i:i+3])
            counts[trigram] += 1
            total += 1

    return counts, total

In [22]:
def sentence_logprob_trigram(sentence, model, delta):
    """
    Log probability of one sentence under a smoothed trigram character LM.
    Unseen characters are mapped to <UNK>.
    """
    chars = [ch if ch in model["vocab"] else "<UNK>" for ch in sentence]
    seq = ["<s>", "<s>"] + chars + ["</s>"]

    logprob = 0.0
    predicted_symbols = 0

    for i in range(2, len(seq)):
        trigram = (seq[i-2], seq[i-1], seq[i])
        context = (seq[i-2], seq[i-1])

        trigram_count = model["trigram_counts"][trigram]
        context_count = model["trigram_context_counts"][context]

        prob = (trigram_count + delta) / (
            context_count + delta * model["output_vocabulary_size"]
        )

        logprob += math.log2(prob)
        predicted_symbols += 1

    return logprob, predicted_symbols


def cross_entropy(sentences, model, delta):
    total_logprob = 0.0
    total_symbols = 0

    for sent in sentences:
        lp, n = sentence_logprob_trigram(sent, model, delta)
        total_logprob += lp
        total_symbols += n

    return -total_logprob / total_symbols


def estimate_delta(heldout_sentences, model):
    """
    Estimate add-delta smoothing parameter on the heldout set.
    We choose the delta with the lowest heldout cross-entropy.
    """
    de?: 

    best_delta = None
    best_ce = float("inf")

    for delta in candidate_deltas:
        ce = cross_entropy(heldout_sentences, model, float(delta))
        if ce < best_ce:
            best_ce = ce
            best_delta = float(delta)

    return best_delta, best_ce

In [24]:
models = {}

for language in LANGUAGES:
    model = build_language_model(splits[language]["train"])
    delta, heldout_ce = estimate_delta(splits[language]["heldout"], model)

    model["delta"] = delta
    model["heldout_cross_entropy"] = heldout_ce
    models[language] = model

print("Training finished.")

Training finished.


In [26]:
# Report 5 most common character trigrams per language.
# Here relative_frequency = count / total number of raw character trigram tokens
# in the TRAINING split.

top_trigram_rows = []

for language in LANGUAGES:
    trigram_counts, total_trigrams = raw_trigram_counts(splits[language]["train"])

    for trigram, count in trigram_counts.most_common(5):
        top_trigram_rows.append({
            "language": language,
            "trigram": repr(trigram),
            "count": count,
            "relative_frequency": count / total_trigrams,
        })

top_trigrams_df = pd.DataFrame(top_trigram_rows)
top_trigrams_df

,language,trigram,count,relative_frequency
0,English,' th',149964,0.014617
1,English,'the',149868,0.014607
2,English,'he ',126578,0.012337
3,English,"' , '",86782,0.008458
4,English,'ed ',76886,0.007494
5,Polish,"' , '",60153,0.007057
6,Polish,'ie ',54218,0.006361
7,Polish,'nie',52287,0.006134
8,Polish,' po',49148,0.005766
9,Polish,' w ',45501,0.005338


In [28]:
# cell 12
print("Top 5 most common character trigrams per language:\n")
print(top_trigrams_df.to_string(index=False))

Top 5 most common character trigrams per language:

 language trigram  count  relative_frequency
  English   ' th' 149964            0.014617
  English   'the' 149868            0.014607
  English   'he ' 126578            0.012337
  English   ' , '  86782            0.008458
  English   'ed '  76886            0.007494
   Polish   ' , '  60153            0.007057
   Polish   'ie '  54218            0.006361
   Polish   'nie'  52287            0.006134
   Polish   ' po'  49148            0.005766
   Polish   ' w '  45501            0.005338
Hungarian   ' , '  92921            0.010170
Hungarian   ' a '  91460            0.010010
Hungarian   'en '  43939            0.004809
Hungarian   ' sz'  43705            0.004783
Hungarian   'tt '  38933            0.004261


In [30]:
# cell 13
delta_rows = []

for language in LANGUAGES:
    delta_rows.append({
        "language": language,
        "best_delta": models[language]["delta"],
        "heldout_cross_entropy": models[language]["heldout_cross_entropy"],
    })

delta_df = pd.DataFrame(delta_rows).sort_values("language")
delta_df

,language,best_delta,heldout_cross_entropy
0,English,0.005,2.917446
2,Hungarian,0.004,3.170511
1,Polish,0.006,3.117247


In [32]:
print("Estimated add-delta smoothing parameters:\n")
print(delta_df.to_string(index=False))

Estimated add-delta smoothing parameters:

 language  best_delta  heldout_cross_entropy
  English       0.005               2.917446
Hungarian       0.004               3.170511
   Polish       0.006               3.117247


In [34]:
# Cross-entropies of all trigram models on all test sets
# Rows = model language
# Columns = test language

ce_matrix = pd.DataFrame(index=LANGUAGES.keys(), columns=LANGUAGES.keys(), dtype=float)

for model_language in LANGUAGES:
    for test_language in LANGUAGES:
        ce = cross_entropy(
            splits[test_language]["test"],
            models[model_language],
            models[model_language]["delta"],
        )
        ce_matrix.loc[model_language, test_language] = ce

ce_matrix

,English,Polish,Hungarian
English,2.919658,6.151378,6.286177
Polish,4.225958,3.116446,6.063083
Hungarian,4.304630,5.906396,3.171149


In [36]:
print("Cross-entropy matrix (rows=model, columns=test set):\n")
print(ce_matrix.to_string())

Cross-entropy matrix (rows=model, columns=test set):

            English    Polish  Hungarian
English    2.919658  6.151378   6.286177
Polish     4.225958  3.116446   6.063083
Hungarian  4.304630  5.906396   3.171149


In [38]:
lang_to_moses = {lang: info["moses"] for lang, info in LANGUAGES.items()}

def identify_language(text):
    """
    Return a list of (probability, language), ordered highest first.

    To avoid numerical underflow on long texts, we:
    1. compute log probabilities under each trigram LM
    2. convert them to normalized probabilities across languages
    """
    cleaned = re.sub(r"\s+", " ", text.strip().lower())
    log_scores = []

    for language, model in models.items():
        tokenizer = MosesTokenizer(lang=lang_to_moses[language])
        tokenized = tokenizer.tokenize(cleaned, return_str=True, escape=False)

        lp, _ = sentence_logprob_trigram(tokenized, model, model["delta"])
        log_scores.append((lp, language))

    max_lp = max(lp for lp, _ in log_scores)
    weights = [(2 ** (lp - max_lp), language) for lp, language in log_scores]
    z = sum(w for w, _ in weights)

    results = [(w / z, language) for w, language in weights]
    results.sort(reverse=True)
    return results

In [56]:
# Examples
print(identify_language("This is a short example written in English."))
print(identify_language("To jest krótkie przykładowe zdanie po polsku."))
print(identify_language("Ez egy rövid magyar példamondat."))

[(0.9999999999999991, 'English'), (9.203465419983743e-16, 'Hungarian'), (4.897353782285313e-21, 'Polish')]
[(1.0, 'Polish'), (2.9741254640338556e-40, 'Hungarian'), (6.779270211582302e-41, 'English')]
[(1.0, 'Hungarian'), (1.1136121381086933e-24, 'English'), (7.2251240368678424e-31, 'Polish')]


In [58]:
# nicer display with rounded numbers
pretty_ce = ce_matrix.copy()
for row in pretty_ce.index:
    for col in pretty_ce.columns:
        pretty_ce.loc[row, col] = round(pretty_ce.loc[row, col], 4)

pretty_delta = delta_df.copy()
pretty_delta["best_delta"] = pretty_delta["best_delta"].round(4)
pretty_delta["heldout_cross_entropy"] = pretty_delta["heldout_cross_entropy"].round(4)

pretty_top = top_trigrams_df.copy()
pretty_top["relative_frequency"] = pretty_top["relative_frequency"].round(6)

print("=== DATA SIZES ===")
print(sizes_df.to_string(index=False))
print("\n=== TOP 5 CHARACTER TRIGRAMS ===")
print(pretty_top.to_string(index=False))
print("\n=== SMOOTHING PARAMETERS ===")
print(pretty_delta.to_string(index=False))
print("\n=== CROSS-ENTROPY MATRIX ===")
print(pretty_ce.to_string())

=== DATA SIZES ===
 language  sentences  tokens    bytes
  English     100000 2387805 13149854
Hungarian     100000 1833664 12830423
   Polish     100000 1751344 11563037

=== TOP 5 CHARACTER TRIGRAMS ===
 language trigram  count  relative_frequency
  English   ' th' 149964            0.014617
  English   'the' 149868            0.014607
  English   'he ' 126578            0.012337
  English   ' , '  86782            0.008458
  English   'ed '  76886            0.007494
   Polish   ' , '  60153            0.007057
   Polish   'ie '  54218            0.006361
   Polish   'nie'  52287            0.006134
   Polish   ' po'  49148            0.005766
   Polish   ' w '  45501            0.005338
Hungarian   ' , '  92921            0.010170
Hungarian   ' a '  91460            0.010010
Hungarian   'en '  43939            0.004809
Hungarian   ' sz'  43705            0.004783
Hungarian   'tt '  38933            0.004261

=== SMOOTHING PARAMETERS ===
 language  best_delta  heldout_cross_entropy


In [78]:
# Aditionally report:
# Percentage of heldout OOV tokens
# Percentage of test OOV tokens
    
def oov_ratio_against_train(train_sentences, eval_sentences):
    
    train_vocab = set()
    for sent in train_sentences:
        train_vocab.update(sent.split())

    eval_tokens = []
    for sent in eval_sentences:
        eval_tokens.extend(sent.split())

    total_eval_tokens = len(eval_tokens)
    oov_eval_tokens = sum(1 for tok in eval_tokens if tok not in train_vocab)
    oov_ratio = oov_eval_tokens / total_eval_tokens if total_eval_tokens > 0 else 0.0

    return total_eval_tokens, oov_eval_tokens, oov_ratio


oov_rows = []

for language in LANGUAGES:
    train_sents = splits[language]["train"]
    heldout_sents = splits[language]["heldout"]
    test_sents = splits[language]["test"]

    heldout_total, heldout_oov, heldout_ratio = oov_ratio_against_train(train_sents, heldout_sents)
    test_total, test_oov, test_ratio = oov_ratio_against_train(train_sents, test_sents)

    oov_rows.append({
        "language": language,
        "heldout_tokens": heldout_total,
        "heldout_oov_tokens": heldout_oov,
        "heldout_oov_ratio": heldout_ratio,
        "test_tokens": test_total,
        "test_oov_tokens": test_oov,
        "test_oov_ratio": test_ratio,
    })

oov_df = pd.DataFrame(oov_rows)
display(oov_df)

heldout_oov_form = ", ".join(
    f"{oov_df.set_index('language').loc[language, 'heldout_oov_ratio']:.6f}"
    for language in LANGUAGES
)

test_oov_form = ", ".join(
    f"{oov_df.set_index('language').loc[language, 'test_oov_ratio']:.6f}"
    for language in LANGUAGES
)

print("Percentage of heldout OOV tokens:")
print(heldout_oov_form)

print("\nPercentage of test OOV tokens:")
print(test_oov_form)

,language,heldout_tokens,heldout_oov_tokens,heldout_oov_ratio,test_tokens,test_oov_tokens,test_oov_ratio
0,English,239267,6880,0.028754,238892,6700,0.028046
1,Polish,176624,12953,0.073337,175658,12894,0.073404
2,Hungarian,182814,17894,0.097881,184076,18112,0.098394


Percentage of heldout OOV tokens:
0.028754, 0.073337, 0.097881

Percentage of test OOV tokens:
0.028046, 0.073404, 0.098394
